# Đề tài 01 — Frozen Lake 


In [ ]:
import random
from collections import defaultdict
import numpy as np

ACTIONS = [0, 1, 2, 3]
DELTA = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}
ARROWS = {0: '←', 1: '↓', 2: '→', 3: '↑'}
PERP = {0: [1, 3], 1: [0, 2], 2: [1, 3], 3: [0, 2]}


## 1. Môi trường `FrozenLakeEnv`


In [ ]:
class FrozenLakeEnv:
    def __init__(self, rows, cols, start, goal, holes,
                 default_slip=0.0, slip_map=None, boots_tiles=None,
                 step_reward=-0.01, goal_reward=1.0, hole_reward=-1.0, max_steps=200):
        self.rows, self.cols = rows, cols
        self.start, self.goal = start, goal
        self.holes = set(holes)
        self.default_slip = default_slip
        self.slip_map = slip_map or {}
        self.boots_tiles = set(boots_tiles) if boots_tiles else set()
        self.uses_boots = len(self.boots_tiles) > 0
        self.step_reward = step_reward
        self.goal_reward = goal_reward
        self.hole_reward = hole_reward
        self.max_steps = max_steps
        self.pos = self.start
        self.k = 0
        self.t = 0

    def clip(self, r, c):
        return max(0, min(self.rows - 1, r)), max(0, min(self.cols - 1, c))

    def is_terminal_cell(self, cell):
        return cell in self.holes or cell == self.goal

    def reset(self):
        self.pos, self.k, self.t = self.start, 0, 0
        return (self.pos[0], self.pos[1], self.k) if self.uses_boots else (self.pos[0], self.pos[1])

    def step(self, action):
        self.t += 1
        slip = 0.0 if self.k > 0 else self.slip_map.get(self.pos, self.default_slip)
        
        actual = random.choice(PERP[action]) if (slip > 0 and random.random() < slip) else action
        dr, dc = DELTA[actual]
        r, c = self.clip(self.pos[0] + dr, self.pos[1] + dc)
       
        if self.uses_boots:
            self.k = 3 if (r, c) in self.boots_tiles else max(self.k - 1, 0)
        
        done = (r, c) in self.holes or (r, c) == self.goal or self.t >= self.max_steps
        reward = self.hole_reward if (r, c) in self.holes else (self.goal_reward if (r, c) == self.goal else self.step_reward)
        self.pos = (r, c)
        next_s = (r, c, self.k) if self.uses_boots else (r, c)
        return next_s, reward, done, {}

    def all_states(self):
        cells = [(r, c) for r in range(self.rows) for c in range(self.cols)]
        return [(r, c, k) for (r, c) in cells for k in range(4)] if self.uses_boots else cells

    def build_P(self):
        P = {}
        for s in self.all_states():
            pos, k = (s[0], s[1]), (s[2] if self.uses_boots else 0)
            P[s] = {}
            
            if self.is_terminal_cell(pos):
                for a in ACTIONS: P[s][a] = [(1.0, s, 0.0, True)]
                continue
            
            slip = 0.0 if k > 0 else self.slip_map.get(pos, self.default_slip)
            for a in ACTIONS:
                outcomes = [(1 - slip, a)] + [(slip / 2, pa) for pa in PERP[a]] if slip > 0 else [(1.0, a)]
                transitions = []
                
                for prob, act in outcomes:
                    dr, dc = DELTA[act]
                    r, c = self.clip(pos[0] + dr, pos[1] + dc)
                    k2 = (3 if (r, c) in self.boots_tiles else max(k - 1, 0)) if self.uses_boots else 0
                    done = (r, c) in self.holes or (r, c) == self.goal
                    reward = self.hole_reward if (r, c) in self.holes else (self.goal_reward if (r, c) == self.goal else self.step_reward)
                    ns = (r, c, k2) if self.uses_boots else (r, c)
                    transitions.append((prob, ns, reward, done))
                P[s][a] = transitions
        return P

    def render(self):
        grid = [['.' for _ in range(self.cols)] for _ in range(self.rows)]
        for (r, c) in self.holes: grid[r][c] = 'H'
        for (r, c) in self.boots_tiles: grid[r][c] = 'B'
        grid[self.start[0]][self.start[1]] = 'S'
        grid[self.goal[0]][self.goal[1]] = 'G'
        if self.pos not in (self.start, self.goal) and self.pos not in self.holes:
            grid[self.pos[0]][self.pos[1]] = 'A'
        for row in grid:
            print(' '.join(f'{cell:>2}' for cell in row))
        print()


## 2. Thiết lập 4 Setting môi trường


In [28]:
def make_setting1():
    return FrozenLakeEnv(5, 5, (0, 0), (4, 4), [(1, 1), (2, 3), (3, 1)], default_slip=0.0)

def make_setting2(slip=0.3):
    return FrozenLakeEnv(8, 8, (0, 0), (7, 7), [(2, 3), (3, 5), (5, 1), (6, 6)], default_slip=slip)

def make_setting3():
    slip_map = {(r, c): 0.1 if r < 4 else (0.35 if r < 7 else 0.6) for r in range(10) for c in range(10)}
    holes = [(2, 2), (2, 7), (4, 4), (5, 8), (7, 1), (7, 6), (8, 3)]
    return FrozenLakeEnv(10, 10, (0, 0), (9, 9), holes, default_slip=0.2, slip_map=slip_map)

def make_setting4(slip=0.4):
    return FrozenLakeEnv(8, 8, (0, 0), (7, 7), [(2, 3), (3, 5), (5, 1), (6, 6)], default_slip=slip, boots_tiles=[(1, 4)])

env1 = make_setting1()
print("Setting 1 (5x5 Deterministic):")
env1.render()
states_env1 = env1.all_states()
print(states_env1)

env2 = make_setting2()
print("Setting 2 (8x8 Stochastic):")
env2.render()
states_env2 = env2.all_states()
print(states_env2)

env3 = make_setting3()
print("Setting 3 (10x10 with varying slip probabilities):")
env3.render()
states_env3 = env3.all_states()
print(states_env3)

env4 = make_setting4()
print("Setting 4 (8x8 with boots):")
env4.render()
states_env4 = env4.all_states()
print(states_env4[:12])

Setting 1 (5x5 Deterministic):
 S  .  .  .  .
 .  H  .  .  .
 .  .  .  H  .
 .  H  .  .  .
 .  .  .  .  G

[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]
Setting 2 (8x8 Stochastic):
 S  .  .  .  .  .  .  .
 .  .  .  .  .  .  .  .
 .  .  .  H  .  .  .  .
 .  .  .  .  .  H  .  .
 .  .  .  .  .  .  .  .
 .  H  .  .  .  .  .  .
 .  .  .  .  .  .  H  .
 .  .  .  .  .  .  .  G

[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (3, 6), (3, 7), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (4, 7), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6), (6, 7), (7, 0), (7, 1), (7, 2)

## 3.  6 thuật toán RL 

In [8]:
# --- 1. VALUE ITERATION ---
def value_iteration(env, gamma=0.99, theta=1e-8):
    """Value Iteration: tự tính mô hình chuyển trạng thái, lặp Bellman và trả về (V, policy)."""
    P = env.build_P()
    states = env.all_states()
    V = {s: 0.0 for s in states}
    while True:
        delta = 0.0
        for s in states:
            if env.is_terminal_cell((s[0], s[1])): continue
            q_vals = [sum(p * (r + gamma * V[ns] * (not d)) for p, ns, r, d in P[s][a]) for a in ACTIONS]
            best_v = max(q_vals)
            delta = max(delta, abs(best_v - V[s]))
            V[s] = best_v
        if delta < theta: break
    policy = {s: int(np.argmax([sum(p * (r + gamma * V[ns] * (not d)) for p, ns, r, d in P[s][a]) for a in ACTIONS]))
              for s in states if not env.is_terminal_cell((s[0], s[1]))}
    return V, policy


In [ ]:
# --- 2. POLICY ITERATION ---
def policy_iteration(env, gamma=0.99, theta=1e-8):
    P = env.build_P()
    states = env.all_states()
    V = {s: 0.0 for s in states}
    policy = {s: random.choice(ACTIONS) for s in states if not env.is_terminal_cell((s[0], s[1]))}
    while True:
        # Evaluation
        while True:
            delta = 0.0
            for s in states:
                if env.is_terminal_cell((s[0], s[1])): continue
                v_new = sum(p * (r + gamma * V[ns] * (not d)) for p, ns, r, d in P[s][policy[s]])
                delta = max(delta, abs(v_new - V[s]))
                V[s] = v_new
            if delta < theta: break
        # Improvement
        stable = True
        for s in states:
            if env.is_terminal_cell((s[0], s[1])): continue
            old_a = policy[s]
            best_a = int(np.argmax([sum(p * (r + gamma * V[ns] * (not d)) for p, ns, r, d in P[s][a]) for a in ACTIONS]))
            if best_a != old_a: stable = False
            policy[s] = best_a
        if stable: break
    return V, policy


In [ ]:
# --- 3. Q-LEARNING ---
def q_learning(env, episodes=2000, alpha=0.2, gamma=0.99, eps_start=1.0, eps_min=0.05, seed=0):
    Q = defaultdict(lambda: np.zeros(len(ACTIONS)))
    decay = int(episodes * 0.8)
    for ep in range(episodes):
        eps = max(eps_min, eps_start - (eps_start - eps_min) * min(1.0, ep / decay))
        s, done = env.reset(), False
        while not done:
            a = random.choice(ACTIONS) if random.random() < eps else int(np.argmax(Q[s]))
            s2, r, done, _ = env.step(a)
            best_next_q = np.max(Q[s2]) if not done else 0.0
            Q[s][a] += alpha * (r + gamma * best_next_q - Q[s][a])
            s = s2
    policy = {s: int(np.argmax(Q[s])) for s in Q}
    return Q, policy


In [ ]:
# --- 4. SARSA ---
def sarsa(env, episodes=2000, alpha=0.2, gamma=0.99, eps_start=1.0, eps_min=0.05, seed=0):
    random.seed(seed); np.random.seed(seed)
    Q = defaultdict(lambda: np.zeros(len(ACTIONS)))
    decay = int(episodes * 0.8)
    for ep in range(episodes):
        eps = max(eps_min, eps_start - (eps_start - eps_min) * min(1.0, ep / decay))
        s = env.reset()
        a = random.choice(ACTIONS) if random.random() < eps else int(np.argmax(Q[s]))
        done = False
        while not done:
            s2, r, done, _ = env.step(a)
            a2 = (random.choice(ACTIONS) if random.random() < eps else int(np.argmax(Q[s2]))) if not done else 0
            next_q = Q[s2][a2] if not done else 0.0
            Q[s][a] += alpha * (r + gamma * next_q - Q[s][a])
            s, a = s2, a2
    policy = {s: int(np.argmax(Q[s])) for s in Q}
    return Q, policy


In [ ]:
# --- 5. MONTE CARLO ---
def monte_carlo(env, episodes=2000, alpha=0.1, gamma=0.99, eps_start=1.0, eps_min=0.05, seed=0):
    random.seed(seed); np.random.seed(seed)
    Q = defaultdict(lambda: np.zeros(len(ACTIONS)))
    decay = int(episodes * 0.8)
    for ep in range(episodes):
        eps = max(eps_min, eps_start - (eps_start - eps_min) * min(1.0, ep / decay))
        s, done = env.reset(), False
        trajectory = []
        while not done:
            a = random.choice(ACTIONS) if random.random() < eps else int(np.argmax(Q[s]))
            s2, r, done, _ = env.step(a)
            trajectory.append((s, a, r))
            s = s2
        G = 0.0
        for s, a, r in reversed(trajectory):
            G = gamma * G + r
            Q[s][a] += alpha * (G - Q[s][a])
    policy = {s: int(np.argmax(Q[s])) for s in Q}
    return Q, policy


In [ ]:
# --- 6. TD-LEARNING (Expected SARSA TD Control) ---
def td_learning(env, episodes=2000, alpha=0.2, gamma=0.99, eps_start=1.0, eps_min=0.05, seed=0):
    random.seed(seed); np.random.seed(seed)
    Q = defaultdict(lambda: np.zeros(len(ACTIONS)))
    decay = int(episodes * 0.8)
    num_a = len(ACTIONS)
    for ep in range(episodes):
        eps = max(eps_min, eps_start - (eps_start - eps_min) * min(1.0, ep / decay))
        s, done = env.reset(), False
        while not done:
            a = random.choice(ACTIONS) if random.random() < eps else int(np.argmax(Q[s]))
            s2, r, done, _ = env.step(a)
            if not done:
                best_a = int(np.argmax(Q[s2]))
                expected_q = sum(Q[s2][act] * (1.0 - eps + eps/num_a if act == best_a else eps/num_a) for act in ACTIONS)
            else:
                expected_q = 0.0
            Q[s][a] += alpha * (r + gamma * expected_q - Q[s][a])
            s = s2
    policy = {s: int(np.argmax(Q[s])) for s in Q}
    return Q, policy


## 4. Test & So sánh hiệu năng 6 thuật toán


In [14]:
import time

def evaluate_agent(env, policy, test_episodes=200):
    successes, total_rewards, total_steps = 0, [], []
    for _ in range(test_episodes):
        s, done, ep_reward, steps = env.reset(), False, 0.0, 0
        while not done:
            a = policy.get(s, 0)
            s, r, done, _ = env.step(a)
            ep_reward += r
            steps += 1
            if r == env.goal_reward:
                successes += 1
        total_rewards.append(ep_reward)
        total_steps.append(steps)
    return (successes / test_episodes) * 100, np.mean(total_rewards), np.mean(total_steps)

# Thử nghiệm Setting 1: 5x5 Deterministic (Bản đồ xác định, không trượt)
env1 = make_setting1()
algos_setting1 = [
    ("Value Iteration", lambda: value_iteration(env1)),
    ("Policy Iteration", lambda: policy_iteration(env1)),
    ("Q-Learning", lambda: q_learning(env1, episodes=2500, seed=42)),
    ("SARSA", lambda: sarsa(env1, episodes=2500, seed=42)),
    ("Monte Carlo", lambda: monte_carlo(env1, episodes=2500, seed=42)),
    ("TD-Learning", lambda: td_learning(env1, episodes=2500, seed=42)),
]

print("=== KẾT QUẢ THỬ NGHIỆM TRÊN SETTING 1 (5x5 DETERMINISTIC) ===")
print(f"{'Thuật toán':<22} | {'Tỷ lệ thắng (%)':<15} | {'Reward TB':<12} | {'Số bước TB':<12} | {'Thời gian (s)':<12}")
print("-" * 82)

policies1 = {}
for name, train_fn in algos_setting1:
    t0 = time.time()
    _, pol = train_fn()
    t_train = time.time() - t0
    policies1[name] = pol
    win_rate, avg_r, avg_s = evaluate_agent(env1, pol, test_episodes=200)
    print(f"{name:<22} | {win_rate:>13.1f}% | {avg_r:>12.3f} | {avg_s:>12.2f} | {t_train:>11.4f}s")


=== KẾT QUẢ THỬ NGHIỆM TRÊN SETTING 1 (5x5 DETERMINISTIC) ===
Thuật toán             | Tỷ lệ thắng (%) | Reward TB    | Số bước TB   | Thời gian (s)
----------------------------------------------------------------------------------
Value Iteration        |         100.0% |        0.930 |         8.00 |      0.0010s
Policy Iteration       |         100.0% |        0.930 |         8.00 |      0.0128s
Q-Learning             |         100.0% |        0.930 |         8.00 |      0.0784s
SARSA                  |         100.0% |        0.930 |         8.00 |      0.0395s
Monte Carlo            |         100.0% |        0.930 |         8.00 |      0.0363s
TD-Learning            |         100.0% |        0.930 |         8.00 |      0.0622s


In [ ]:
# Thử nghiệm Setting 2: 8x8 
env2 = make_setting2(slip=0.3)
algos_setting2 = [
    ("Value Iteration", lambda: value_iteration(env2)),
    ("Policy Iteration", lambda: policy_iteration(env2)),
    ("Q-Learning", lambda: q_learning(env2, episodes=4000, seed=42)),
    ("SARSA", lambda: sarsa(env2, episodes=4000, seed=42)),
    ("Monte Carlo", lambda: monte_carlo(env2, episodes=4000, seed=42)),
    ("TD-Learning", lambda: td_learning(env2, episodes=4000, seed=42)),
]

print("=== KẾT QUẢ THỬ NGHIỆM TRÊN SETTING 2 (8x8 SLIPPERY slip=0.3) ===")
print(f"{'Thuật toán':<22} | {'Tỷ lệ thắng (%)':<15} | {'Reward TB':<12} | {'Số bước TB':<12} | {'Thời gian (s)':<12}")
print("-" * 82)

policies2 = {}
for name, train_fn in algos_setting2:
    t0 = time.time()
    _, pol = train_fn()
    t_train = time.time() - t0
    policies2[name] = pol
    win_rate, avg_r, avg_s = evaluate_agent(env2, pol, test_episodes=200)
    print(f"{name:<22} | {win_rate:>13.1f}% | {avg_r:>12.3f} | {avg_s:>12.2f} | {t_train:>11.4f}s")

# In bản đồ chính sách của thuật toán tiêu biểu
def print_policy(env, policy, title="Policy Grid"):
    print(f"\n=== {title} ===")
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            pos = (r, c)
            s = (r, c, 0) if env.uses_boots else (r, c)
            if pos == env.start: row.append(f"S({ARROWS.get(policy.get(s, -1), '?')})")
            elif pos == env.goal: row.append(" G  ")
            elif pos in env.holes: row.append(" H  ")
            elif pos in env.boots_tiles: row.append(f"B({ARROWS.get(policy.get(s, -1), '?')})")
            else: row.append(f" {ARROWS.get(policy.get(s, -1), '.')}  ")
        print(" ".join(row))

print_policy(env2, policies2["Value Iteration"], "Chính sách tối ưu Setting 2 (Value Iteration - Né hố an toàn)")


=== KẾT QUẢ THỬ NGHIỆM TRÊN SETTING 2 (8x8 SLIPPERY slip=0.3) ===
Thuật toán             | Tỷ lệ thắng (%) | Reward TB    | Số bước TB   | Thời gian (s)
----------------------------------------------------------------------------------
Value Iteration        |         100.0% |        0.717 |        29.29 |      0.0137s
Policy Iteration       |         100.0% |        0.716 |        29.36 |      0.0308s
Q-Learning             |         100.0% |        0.696 |        31.38 |      0.3855s
SARSA                  |         100.0% |        0.692 |        31.79 |      0.2296s
Monte Carlo            |          65.0% |        0.105 |        20.48 |      0.2036s
TD-Learning            |         100.0% |        0.701 |        30.91 |      0.3684s

=== Chính sách tối ưu Setting 2 (Value Iteration - Né hố an toàn) ===
S(→)  →    →    →    →    →    ↓    ↓  
 ↓    ↓    →    ↑    →    →    ↓    ↓  
 ↓    ↓    ←    H    →    ↑    →    ↓  
 →    →    ↓    ↓    ←    H    →    ↓  
 →    ↑    →    ↓    ↓ 

In [16]:
# Thử nghiệm Setting 3: 10x10 Multi-zone (3 vùng ma sát: 0.1, 0.35, 0.6)
env3 = make_setting3()
algos_setting3 = [
    ("Value Iteration", lambda: value_iteration(env3)),
    ("Policy Iteration", lambda: policy_iteration(env3)),
    ("Q-Learning", lambda: q_learning(env3, episodes=3500, seed=42)),
    ("SARSA", lambda: sarsa(env3, episodes=3500, seed=42)),
    ("Monte Carlo", lambda: monte_carlo(env3, episodes=3500, seed=42)),
    ("TD-Learning", lambda: td_learning(env3, episodes=3500, seed=42)),
]

print("=== KẾT QUẢ THỬ NGHIỆM TRÊN SETTING 3 (10x10 MULTI-ZONE SLIPPERY) ===")
print(f"{'Thuật toán':<22} | {'Tỷ lệ thắng (%)':<15} | {'Reward TB':<12} | {'Số bước TB':<12} | {'Thời gian (s)':<12}")
print("-" * 82)

policies3 = {}
for name, train_fn in algos_setting3:
    t0 = time.time()
    _, pol = train_fn()
    t_train = time.time() - t0
    policies3[name] = pol
    win_rate, avg_r, avg_s = evaluate_agent(env3, pol, test_episodes=200)
    print(f"{name:<22} | {win_rate:>13.1f}% | {avg_r:>12.3f} | {avg_s:>12.2f} | {t_train:>11.4f}s")

print_policy(env3, policies3["Value Iteration"], "Chính sách Setting 3 (Value Iteration - Né vùng băng mỏng)")


=== KẾT QUẢ THỬ NGHIỆM TRÊN SETTING 3 (10x10 MULTI-ZONE SLIPPERY) ===
Thuật toán             | Tỷ lệ thắng (%) | Reward TB    | Số bước TB   | Thời gian (s)
----------------------------------------------------------------------------------
Value Iteration        |         100.0% |        0.650 |        35.97 |      0.0230s
Policy Iteration       |         100.0% |        0.646 |        36.44 |      0.0931s
Q-Learning             |         100.0% |        0.637 |        37.34 |      0.3478s
SARSA                  |         100.0% |        0.646 |        36.40 |      0.2114s
Monte Carlo            |          16.5% |       -1.488 |       128.81 |      0.2423s
TD-Learning            |          96.5% |        0.603 |        33.74 |      0.3521s

=== Chính sách Setting 3 (Value Iteration - Né vùng băng mỏng) ===
S(→)  →    →    →    →    →    →    →    →    ↓  
 ↑    ↑    ↑    →    →    →    →    ↑    →    ↓  
 ↑    ←    H    →    →    ↓    ←    H    →    ↓  
 →    →    ↓    →    ↑    →    →

In [17]:
# Thử nghiệm Setting 4: 8x8 Nút thắt cổ chai & Ô giày chống trượt B
env4 = make_setting4(slip=0.4)
algos_setting4 = [
    ("Value Iteration", lambda: value_iteration(env4)),
    ("Policy Iteration", lambda: policy_iteration(env4)),
    ("Q-Learning", lambda: q_learning(env4, episodes=3500, seed=42)),
    ("SARSA", lambda: sarsa(env4, episodes=3500, seed=42)),
    ("Monte Carlo", lambda: monte_carlo(env4, episodes=3500, seed=42)),
    ("TD-Learning", lambda: td_learning(env4, episodes=3500, seed=42)),
]

print("=== KẾT QUẢ THỬ NGHIỆM TRÊN SETTING 4 (8x8 NÚT THẮT CỔ CHAI & GIÀY B) ===")
print(f"{'Thuật toán':<22} | {'Tỷ lệ thắng (%)':<15} | {'Reward TB':<12} | {'Số bước TB':<12} | {'Thời gian (s)':<12}")
print("-" * 82)

policies4 = {}
for name, train_fn in algos_setting4:
    t0 = time.time()
    _, pol = train_fn()
    t_train = time.time() - t0
    policies4[name] = pol
    win_rate, avg_r, avg_s = evaluate_agent(env4, pol, test_episodes=200)
    print(f"{name:<22} | {win_rate:>13.1f}% | {avg_r:>12.3f} | {avg_s:>12.2f} | {t_train:>11.4f}s")

print_policy(env4, policies4["Value Iteration"], "Chính sách Setting 4 (Value Iteration - Ghé ăn giày B để qua khe hẹp)")


=== KẾT QUẢ THỬ NGHIỆM TRÊN SETTING 4 (8x8 NÚT THẮT CỔ CHAI & GIÀY B) ===
Thuật toán             | Tỷ lệ thắng (%) | Reward TB    | Số bước TB   | Thời gian (s)
----------------------------------------------------------------------------------
Value Iteration        |         100.0% |        0.691 |        31.95 |      0.0467s
Policy Iteration       |         100.0% |        0.679 |        33.11 |      0.1391s
Q-Learning             |         100.0% |        0.668 |        34.19 |      0.3695s
SARSA                  |         100.0% |        0.497 |        51.27 |      0.2612s
Monte Carlo            |          30.5% |       -0.801 |        63.91 |      0.1820s
TD-Learning            |          96.0% |        0.248 |        72.17 |      0.3945s

=== Chính sách Setting 4 (Value Iteration - Ghé ăn giày B để qua khe hẹp) ===
S(→)  →    →    →    ↓    →    →    ↓  
 →    →    →    ↑   B(→)  →    ↓    ↓  
 ↓    ↓    ←    H    →    ↑    →    ↓  
 →    →    ↓    ↓    ←    H    →    ↓  
 →    ↑

## 5. Trực quan hóa Bảng Value và Bảng Policy (Trước vs Sau khi chạy)

So sánh sự thay đổi của:
1. **Bảng Giá trị trạng thái $V(s)$**:
   - *Trước khi chạy*: Khởi tạo bằng $0.000$ trên mọi ô (chưa có thông tin về Goal hay Hole).
   - *Sau khi chạy*: Giá trị tối ưu $V^*(s)$ lan truyền từ Goal ($1.000$) ngược về Start ($0.864$), phản ánh phần thưởng kỳ vọng tích lũy.
2. **Bảng Chính sách $\pi(s)$**:
   - *Trước khi chạy*: Chính sách ngẫu nhiên hoặc chưa xác định.
   - *Sau khi chạy*: Từng ô có mũi tên chỉ dẫn chính xác hướng đi ngắn nhất và an toàn né hố.


In [18]:
# Định nghĩa hàm in bảng Giá trị V(s) và Chính sách Policy
def print_value_table(env, V, title="Bảng Giá Trị V(s)"):
    print(f"=== {title} ===")
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            pos = (r, c)
            if pos in env.holes: row.append("   H   ")
            elif pos == env.goal: row.append("   G   ")
            else:
                s = (r, c, 0) if env.uses_boots else (r, c)
                row.append(f"{V.get(s, 0.0):6.3f} ")
        print(" ".join(row))
    print()

def print_policy_table(env, policy, title="Bảng Chính Sách Policy"):
    print(f"=== {title} ===")
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            pos = (r, c)
            s = (r, c, 0) if env.uses_boots else (r, c)
            if pos == env.start:
                arrow = ARROWS.get(policy.get(s, -1), '?')
                row.append(f"S({arrow})")
            elif pos == env.goal: row.append(" G  ")
            elif pos in env.holes: row.append(" H  ")
            elif pos in env.boots_tiles:
                arrow = ARROWS.get(policy.get(s, -1), '?')
                row.append(f"B({arrow})")
            else:
                arrow = ARROWS.get(policy.get(s, -1), '.')
                row.append(f" {arrow}  ")
        print(" ".join(row))
    print()

env = make_setting1()

# ----------------- GIAI ĐOẠN 1: TRƯỚC KHI BẮT ĐẦU CHẠY -----------------
random.seed(42)
V_before = {s: 0.0 for s in env.all_states()}
policy_before = {s: random.choice(ACTIONS) for s in env.all_states() if not env.is_terminal_cell((s[0], s[1]))}

print_value_table(env, V_before, "GIAI ĐOẠN 1: BẢNG VALUE TRƯỚC KHI CHẠY (V0 = 0.0)")
print_policy_table(env, policy_before, "GIAI ĐOẠN 1: BẢNG POLICY TRƯỚC KHI CHẠY (NGẪU NHIÊN)")


=== GIAI ĐOẠN 1: BẢNG VALUE TRƯỚC KHI CHẠY (V0 = 0.0) ===
 0.000   0.000   0.000   0.000   0.000 
 0.000     H     0.000   0.000   0.000 
 0.000   0.000   0.000     H     0.000 
 0.000     H     0.000   0.000   0.000 
 0.000   0.000   0.000   0.000     G   

=== GIAI ĐOẠN 1: BẢNG POLICY TRƯỚC KHI CHẠY (NGẪU NHIÊN) ===
S(←)  ←    →    ↓    ↓  
 ↓    H    ←    ←    ↑  
 ←    ←    ←    H    ↓  
 ↓    H    ←    ↓    ↑  
 ↓    ↑    →    ←    G  



In [19]:
# ----------------- GIAI ĐOẠN 2: SAU KHI CHẠY VALUE ITERATION -----------------
V_vi, policy_vi = value_iteration(env)

print_value_table(env, V_vi, "GIAI ĐOẠN 2: BẢNG VALUE TỐI ƯU (VALUE ITERATION V*)")
print_policy_table(env, policy_vi, "GIAI ĐOẠN 2: BẢNG POLICY TỐI ƯU (VALUE ITERATION π*)")


=== GIAI ĐOẠN 2: BẢNG VALUE TỐI ƯU (VALUE ITERATION V*) ===
 0.864   0.883   0.902   0.921   0.941 
 0.883     H     0.921   0.941   0.960 
 0.902   0.921   0.941     H     0.980 
 0.921     H     0.960   0.980   1.000 
 0.941   0.960   0.980   1.000     G   

=== GIAI ĐOẠN 2: BẢNG POLICY TỐI ƯU (VALUE ITERATION π*) ===
S(↓)  →    ↓    ↓    ↓  
 ↓    H    ↓    →    ↓  
 ↓    →    ↓    H    ↓  
 ↓    H    ↓    ↓    ↓  
 →    →    →    →    G  



In [20]:
# ----------------- GIAI ĐOẠN 3: SAU KHI HỌC Q-LEARNING -----------------
Q_ql, policy_ql = q_learning(env, episodes=2500, seed=42)
V_ql = {s: float(np.max(Q_ql[s])) for s in env.all_states()}

print_value_table(env, V_ql, "GIAI ĐOẠN 3: BẢNG VALUE HỌC ĐƯỢC TỪ BẢNG Q (max Q)")
print_policy_table(env, policy_ql, "GIAI ĐOẠN 3: BẢNG POLICY HỌC ĐƯỢC (Q-LEARNING)")


=== GIAI ĐOẠN 3: BẢNG VALUE HỌC ĐƯỢC TỪ BẢNG Q (max Q) ===
 0.864   0.845   0.888   0.654   0.801 
 0.883     H     0.919   0.739   0.893 
 0.902   0.921   0.941     H     0.952 
 0.921     H     0.960   0.980   1.000 
 0.941   0.960   0.980   1.000     G   

=== GIAI ĐOẠN 3: BẢNG POLICY HỌC ĐƯỢC (Q-LEARNING) ===
S(↓)  ←    ↓    →    ↓  
 ↓    H    ↓    →    ↓  
 ↓    →    ↓    H    ↓  
 ↓    H    →    ↓    ↓  
 →    →    →    →    G  



## 6. Thử nghiệm Cấu hình Reward (Reward Shaping) & Phân tích Sự Thay Đổi Policy

**Yêu cầu đề tài:** Thử ít nhất 2 cấu hình reward hoặc tham số môi trường và giải thích policy thay đổi.

Ta thiết kế bản đồ 5×7 có một "bức tường" 3 hố nước chắn ở giữa tại $(1,3), (2,3), (3,3)$ với độ trượt $p = 0.3$:
- **Cấu hình A (Phạt nặng `hole_reward = -1.0`)**: Tác tử có tâm lý *thận trọng (risk-averse)*, né xa các ô hố để đảm bảo an toàn tuyệt đối dù phải đi đường vòng dài hơn.
- **Cấu hình B (Phạt nhẹ `hole_reward = -0.05`)**: Tác tử có tâm lý *mạo hiểm (risk-seeking)*, chấp nhận rủi ro đi sát cạnh mép hố để rút ngắn số bước đi về đích.


In [21]:
# Thử nghiệm 2 cấu hình Reward trên bản đồ 5x7 có tường hố ở giữa
rows_r, cols_r = 5, 7
holes_r = [(1, 3), (2, 3), (3, 3)]
start_r, goal_r = (0, 0), (4, 6)

def make_reward_env(hole_pen):
    return FrozenLakeEnv(rows_r, cols_r, start_r, goal_r, holes_r,
                         default_slip=0.3, step_reward=-0.02, goal_reward=1.0,
                         hole_reward=hole_pen, max_steps=60)

print("=== SO SÁNH 2 CẤU HÌNH PHẠT RƠI HỐ (HOLE REWARD) ===")
for hole_pen in [-1.0, -0.05]:
    env_r = make_reward_env(hole_pen)
    _, pol_r = value_iteration(env_r)
    
    # Đánh giá 500 episodes
    succ, steps_list = 0, []
    for _ in range(500):
        s, done, st = env_r.reset(), False, 0
        while not done:
            s, r, done, _ = env_r.step(pol_r.get(s, 0))
            st += 1
            if r == 1.0: succ += 1
        steps_list.append(st)
        
    tag = "PHẠT NẶNG (Thận trọng)" if hole_pen == -1.0 else "PHẠT NHẸ (Mạo hiểm)"
    print(f"\n>>> Cấu hình hole_reward = {hole_pen} [{tag}]:")
    print(f"    - Tỷ lệ thành công: {succ / 5:.1f}%")
    print(f"    - Số bước trung bình: {np.mean(steps_list):.2f} bước")
    print_policy_table(env_r, pol_r, f"Chính sách khi hole_reward = {hole_pen}")


=== SO SÁNH 2 CẤU HÌNH PHẠT RƠI HỐ (HOLE REWARD) ===

>>> Cấu hình hole_reward = -1.0 [PHẠT NẶNG (Thận trọng)]:
    - Tỷ lệ thành công: 99.0%
    - Số bước trung bình: 22.02 bước
=== Chính sách khi hole_reward = -1.0 ===
S(→)  →    →    ↑    →    ↓    ↓  
 ↓    ↓    ←    H    →    ↓    ↓  
 ↓    ↓    ←    H    →    ↓    ↓  
 ↓    ↓    ←    H    →    →    ↓  
 →    →    →    ↓    →    →    G  


>>> Cấu hình hole_reward = -0.05 [PHẠT NHẸ (Mạo hiểm)]:
    - Tỷ lệ thành công: 81.2%
    - Số bước trung bình: 13.64 bước
=== Chính sách khi hole_reward = -0.05 ===
S(→)  →    →    →    →    ↓    ↓  
 ↓    ↓    ←    H    →    ↓    ↓  
 ↓    ↓    ←    H    →    ↓    ↓  
 ↓    ↓    ←    H    →    →    ↓  
 →    →    →    →    →    →    G  



## 7. Chương trình Demo 1 Episode bằng Policy đã học

**Yêu cầu đề tài:** Chương trình demo một episode bằng policy đã học (sản phẩm tối thiểu).

Hàm `demo_episode` mô phỏng trực quan từng bước chân của Agent từ vạch xuất phát $S$ đến khi về đích $G$, in ra:
- Tọa độ từng bước đi và hành động được chọn.
- Phần thưởng nhận được tại mỗi bước.
- Lộ trình tổng thể và trạng thái hoàn thành.


In [ ]:
# Chương trình Demo một episode chi tiết bằng policy đã học
def demo_episode(env, policy, title="DEMO EPISODE", max_steps=50):
    s = env.reset()
    done = False
    step_num = 0
    total_reward = 0.0
    path = [s[:2] if env.uses_boots else s]
    arrows_name = {0: '← (Trái)', 1: '↓ (Xuống)', 2: '→ (Phải)', 3: '↑ (Lên)'}
    
    print(f"==================================================")
    print(f"   {title}")
    print(f"   Vị trí bắt đầu: {s[:2] if env.uses_boots else s} | Đích đến: {env.goal}")
    print(f"==================================================")
    
    while not done and step_num < max_steps:
        a = policy.get(s, 0)
        s_next, r, done, _ = env.step(a)
        step_num += 1
        total_reward += r
        
        pos_curr = s[:2] if env.uses_boots else s
        pos_next = s_next[:2] if env.uses_boots else s_next
        path.append(pos_next)
        
        print(f"Bước {step_num:02d}: Tại {pos_curr} -> Hành động: {arrows_name.get(a, a):<10} -> Đến {pos_next} | Reward: {r:+.2f}")
        s = s_next
        
    print(f"--------------------------------------------------")
    print(f"Lộ trình di chuyển: {' -> '.join(str(p) for p in path)}")
    print(f"Tổng số bước: {step_num} | Tổng reward: {total_reward:.2f}")
    if r == env.goal_reward:
        print("Trạng thái: VỀ ĐÍCH THÀNH CÔNG! ")
    else:
        print("Trạng thái: THẤT BẠI (Rơi hố hoặc hết bước) ")
    print(f"==================================================\n")

# Chạy demo trên Setting 1 (5x5 Deterministic)
env1 = make_setting1()
_, pol1 = value_iteration(env1)
demo_episode(env1, pol1, "DEMO TRÊN SETTING 1 (5x5 ĐƠN ĐỊNH - 8 BƯỚC TỐI ƯU)")

# Chạy demo trên Setting 4 (8x8 có ô Giày B)
env4 = make_setting4(slip=0.4)
_, pol4 = value_iteration(env4)
demo_episode(env4, pol4, "DEMO TRÊN SETTING 4 (8x8 GHÉ Ô GIÀY B)")


   DEMO TRÊN SETTING 1 (5x5 ĐƠN ĐỊNH - 8 BƯỚC TỐI ƯU)
   Vị trí bắt đầu: (0, 0) | Đích đến: (4, 4)
Bước 01: Tại (0, 0) -> Hành động: ↓ (Xuống)  -> Đến (1, 0) | Reward: -0.01
Bước 02: Tại (1, 0) -> Hành động: ↓ (Xuống)  -> Đến (2, 0) | Reward: -0.01
Bước 03: Tại (2, 0) -> Hành động: ↓ (Xuống)  -> Đến (3, 0) | Reward: -0.01
Bước 04: Tại (3, 0) -> Hành động: ↓ (Xuống)  -> Đến (4, 0) | Reward: -0.01
Bước 05: Tại (4, 0) -> Hành động: → (Phải)   -> Đến (4, 1) | Reward: -0.01
Bước 06: Tại (4, 1) -> Hành động: → (Phải)   -> Đến (4, 2) | Reward: -0.01
Bước 07: Tại (4, 2) -> Hành động: → (Phải)   -> Đến (4, 3) | Reward: -0.01
Bước 08: Tại (4, 3) -> Hành động: → (Phải)   -> Đến (4, 4) | Reward: +1.00
--------------------------------------------------
Lộ trình di chuyển: (0, 0) -> (1, 0) -> (2, 0) -> (3, 0) -> (4, 0) -> (4, 1) -> (4, 2) -> (4, 3) -> (4, 4)
Tổng số bước: 8 | Tổng reward: 0.93
Trạng thái: VỀ ĐÍCH THÀNH CÔNG! 🎉

   DEMO TRÊN SETTING 4 (8x8 GHÉ Ô GIÀY B)
   Vị trí bắt đầu: (0, 0) | Đí